# Pemodelan: AttentionMIL untuk Deteksi Kerusuhan
## UAS Machine Learning - Universitas Dian Nuswantoro

---

## 1. Import Libraries

In [1]:
import sys, os, json, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.dirname(os.getcwd()))

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, confusion_matrix

from core.mil_attention import AttentionMILModel
sns.set_style("whitegrid")
plt.rcParams.update({"font.size": 12})


## 2. Load Model

In [2]:
MODEL_PATH = "../models/mil_final.pt"
DEVICE = "cpu"

model = AttentionMILModel(input_dim=1024, hidden_units=256, dropout=0.3)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE, weights_only=True))
model.eval()
print(f"Model loaded: {sum(p.numel() for p in model.parameters()):,} parameters")


Model loaded: 558,082 parameters


## 3. Load Test Set

In [3]:
with open("../features/final_dataset/metadata.json") as f:
    meta = json.load(f)

test_items = [m for m in meta if m["split"] == "test"]
print(f"Test set: {len(test_items)} videos")

y_true, y_score, y_pred = [], [], []
for item in test_items:
    feat = np.load(item["path"])
    n_seg = min(16, feat.shape[0])
    feat_t = torch.FloatTensor(feat[:n_seg]).unsqueeze(0)
    with torch.no_grad():
        score = torch.sigmoid(model(feat_t)).item()
    y_true.append(item["label"])
    y_score.append(score)
    y_pred.append(1 if score >= 0.5 else 0)

y_true = np.array(y_true)
y_score = np.array(y_score)
y_pred = np.array(y_pred)


Test set: 559 videos


## 4. Performance Metrics

In [4]:
auc = roc_auc_score(y_true, y_score)
f1 = f1_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred)
acc = np.mean(y_true == y_pred)

print(f"{'Metric':15s} | {'Value':>8s}")
print("-" * 26)
print(f"{'AUC':15s} | {auc:.4f}")
print(f"{'Accuracy':15s} | {acc:.4f}")
print(f"{'F1 Score':15s} | {f1:.4f}")
print(f"{'Precision':15s} | {precision:.4f}")
print(f"{'Recall':15s} | {recall:.4f}")
print()
print("Confusion Matrix:")
print(f"{'':16s} Pred Normal  Pred Rusuh")
print(f"{'Actual Normal':15s} {cm[0,0]:5d}     {cm[0,1]:5d}")
print(f"{'Actual Rusuh':15s} {cm[1,0]:5d}     {cm[1,1]:5d}")


Metric          |    Value
--------------------------
AUC             | 0.9563
Accuracy        | 0.8909
F1 Score        | 0.8683
Precision       | 0.8627
Recall          | 0.8739

Confusion Matrix:
                 Pred Normal  Pred Rusuh
Actual Normal     297        32
Actual Rusuh       29       201


## 5. Discussion

**Model Performance Analysis:**

1. **AUC = 0.9563**: Model memiliki kemampuan diskriminasi sangat baik (AUC > 0.9 = excellent). Ini berarti model mampu membedakan video rusuh dan non-rusuh dengan tingkat kepercayaan tinggi.

2. **Accuracy = 89.09%**: Dari 559 video test, 498 video berhasil diklasifikasikan dengan benar.

3. **F1 Score = 0.8683**: Keseimbangan yang baik antara precision dan recall, menunjukkan bahwa model tidak bias ke salah satu kelas.

4. **Precision = 0.8627**: Dari semua video yang diprediksi sebagai rusuh, 86.27% benar-benar rusuh. False positive rate rendah.

5. **Recall = 0.8739**: Dari semua video rusuh yang ada di test set, 87.39% berhasil terdeteksi. False negative rate rendah.

**Confusion Matrix Analysis:**
- 297 video Normal/Damai benar terdeteksi (TN)
- 32 video Normal salah diklasifikasi sebagai Rusuh (FP)
- 201 video Rusuh benar terdeteksi (TP)
- 29 video Rusuh terlewat (FN)

**Kesimpulan:** Model AttentionMIL sangat efektif untuk deteksi kerusuhan dengan False Positive Rate dan False Negative Rate yang rendah.
